In [1]:
# IMPORTS
import os
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from glob import glob
import cv2
import numpy as np
from tqdm import tqdm
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
from PIL import Image

In [2]:

# -----------------------------
# STEM
# -----------------------------
class Stem(nn.Module):
    def __init__(self, in_channels=1, out_channels=64):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 7, stride=2, padding=3),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# -----------------------------
# RESBLOCK
# -----------------------------
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.skip = nn.Identity()

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.skip(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.relu(x + identity)


# -----------------------------
# SE BLOCK
# -----------------------------
class SEBlock(nn.Module):
    def __init__(self, channels, r=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, max(channels // r, 1))
        self.fc2 = nn.Linear(channels // r, channels)

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean(dim=(2, 3))
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y))
        y = y.view(b, c, 1, 1)
        return x * y


# -----------------------------
# ENCODER
# -----------------------------
class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_blocks):
        super().__init__()

        layers = [ResBlock(in_channels, out_channels)]
        for _ in range(1, num_blocks):
            layers.append(ResBlock(out_channels, out_channels))

        self.block = nn.Sequential(*layers)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.block(x)
        skip = x
        x = self.pool(x)
        return x, skip


# -----------------------------
# PIXELSHUFFLE UPSAMPLE
# -----------------------------
class PixelShuffleUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels * 4, 3, padding=1),
            nn.PixelShuffle(2)
        )

    def forward(self, x):
        return self.block(x)


# -----------------------------
# DECODER
# -----------------------------
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = PixelShuffleUp(in_channels, out_channels)

        self.block = nn.Sequential(
            nn.Conv2d(out_channels + skip_channels, out_channels, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            SEBlock(out_channels),
            nn.Conv2d(out_channels, out_channels, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.block(x)
        return x


# -----------------------------
# FULL MODEL
# -----------------------------
class ResSAXUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = Stem(1, 64)

        # Encoder
        self.enc1 = EncoderBlock(64, 128, 3)
        self.enc2 = EncoderBlock(128, 256, 4)
        self.enc3 = EncoderBlock(256, 512, 6)

        # Decoder (FIXED channels)
        self.dec1 = DecoderBlock(512, 512, 256)
        self.dec2 = DecoderBlock(256, 256, 128)
        self.dec3 = DecoderBlock(128, 128, 64)

        # Final upsample (to restore full size)
        self.final_up = PixelShuffleUp(64, 64)

        # Output
        self.final = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)

        x, s1 = self.enc1(x)   # 128
        x, s2 = self.enc2(x)   # 256
        x, s3 = self.enc3(x)   # 512

        # ❌ removed bottleneck
        # x = self.bottleneck(x)

        x = self.dec1(x, s3)
        x = self.dec2(x, s2)
        x = self.dec3(x, s1)

        x = self.final_up(x)
        x = self.final(x)

        return x

In [3]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA Device Name:", torch.cuda.get_device_name(0))

model = ResSAXUNet().to(device)

print("Before summary")

from torchinfo import summary
out = summary(model, input_size=(1, 1, 256, 256), device=device)

print("After summary")
print(out)
# -----------------------
# DATA PREPROCESSING
# -----------------------

# Z SCORE NORM
class PerImageZScore(A.ImageOnlyTransform):
    def __init__(self, p=1.0):
        super().__init__(p=p)

    def apply(self, img, **params):
        img = img.astype(np.float32)
        m = img.mean()
        s = img.std()
        if s < 1e-6:
            s = 1e-6
        return (img - m) / s

class PatchDataset(Dataset):
    def __init__(self, images, masks, patch_size=256, overlap=128, transform=None):
        self.patch_size = patch_size
        self.stride = patch_size - overlap
        self.transform = transform

        # ✅ CACHE IMAGES IN MEMORY
        self.images = [cv2.imread(p, 0) for p in images]
        self.masks = [cv2.imread(p, 0) for p in masks]

        self.patches = []

        for idx in range(len(self.images)):
            img = self.images[idx]
            h, w = img.shape

            for i in range(0, h - patch_size + 1, self.stride):
                for j in range(0, w - patch_size + 1, self.stride):
                    self.patches.append((idx, i, j))

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        img_idx, i, j = self.patches[idx]

        image = self.images[img_idx]
        mask = self.masks[img_idx]

        # PATCH
        image = image[i:i+self.patch_size, j:j+self.patch_size]
        mask = mask[i:i+self.patch_size, j:j+self.patch_size]

        # Binary mask
        mask = (mask > 0).astype(np.float32)

        image = np.expand_dims(image, axis=-1)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # ✅ SAFE CHANNEL HANDLING
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        return image, mask
    
     

# -----------------------
# AUGMENTATIONS
# -----------------------

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=15, p=0.4),
    A.ElasticTransform(alpha=10, sigma=4, p=0.2),
    A.GaussNoise(std_range=(0.02, 0.05), p=0.2),
    A.RandomBrightnessContrast(0.15, 0.15, p=0.4),
    PerImageZScore(p=1.0),
    ToTensorV2()
])
test_transform = A.Compose([
    PerImageZScore(p=1.0),
    ToTensorV2()
])

 
# -----------------------
# LOSS / OPT / SCHEDULER
# -----------------------

def dice_loss(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)

    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))

    dice = (2 * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()


class EnhancedLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits, target):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
        dice = dice_loss(logits, target)
        return 0.3 * bce + 0.7 * dice

optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
loss_fn = EnhancedLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# single global scaler (do not redefine inside train loop)
scaler = torch.amp.GradScaler('cuda')

CUDA Available: True
CUDA Device Name: NVIDIA GeForce RTX 5090
Before summary
After summary
Layer (type:depth-idx)                   Output Shape              Param #
ResSAXUNet                               [1, 1, 256, 256]          --
├─Stem: 1-1                              [1, 64, 128, 128]         --
│    └─Sequential: 2-1                   [1, 64, 128, 128]         --
│    │    └─Conv2d: 3-1                  [1, 64, 128, 128]         3,200
│    │    └─BatchNorm2d: 3-2             [1, 64, 128, 128]         128
│    │    └─ReLU: 3-3                    [1, 64, 128, 128]         --
├─EncoderBlock: 1-2                      [1, 128, 64, 64]          --
│    └─Sequential: 2-2                   [1, 128, 128, 128]        --
│    │    └─ResBlock: 3-4                [1, 128, 128, 128]        230,272
│    │    └─ResBlock: 3-5                [1, 128, 128, 128]        295,680
│    │    └─ResBlock: 3-6                [1, 128, 128, 128]        295,680
│    └─MaxPool2d: 2-3                    [1,

In [1]:
 

# -----------------------
# INFERENCE HELPERS (fixed)
# -----------------------

def create_blend_mask(patch_size, blend_width):
    if blend_width <= 0:
        return np.ones((patch_size, patch_size), dtype=np.float32)
    mask = np.ones((patch_size, patch_size), dtype=np.float32)
    ramp = 0.5 * (1 - np.cos(np.linspace(0, np.pi, blend_width)))
    for i in range(blend_width):
        mask[i, :] *= ramp[i]        # top
        mask[-i-1, :] *= ramp[i]     # bottom
        mask[:, i] *= ramp[i]        # left
        mask[:, -i-1] *= ramp[i]     # right
    return mask

@torch.no_grad()
def predict_single_patch(model, image, device=device, patch_size=256):
    """
    If your image is exactly patch_size x patch_size and you want 1 patch = 1 image.
    image: numpy (H,W) or (H,W,1)
    returns: probability map (H,W)
    """
    model.eval()
    img = image.copy().astype(np.float32)
    if img.ndim == 3:
        img = img[..., 0]
    # normalize using image stats (same as training PerImageZScore)
    m, s = img.mean(), img.std() + 1e-6
    img_norm = (img - m) / s
    tensor = torch.from_numpy(img_norm).float().unsqueeze(0).unsqueeze(0).to(device)  # (1,1,H,W)
    with torch.cuda.amp.autocast(), torch.no_grad():
        logits = model(tensor)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    return prob

@torch.no_grad()
def predict_tiles_no_overlap(model, image, patch_size=256, device=device, batch_size=8):
    """
    Predict using non-overlapping patches of size patch_size.
    Each tile is processed independently (no overlap, no blending).
    Edges are handled with reflect padding so each tile fed to model is patch_size.
    Returns stitched probability map with same H,W as input.
    """
    model.eval()
    H, W = image.shape[:2]
    n_h = int(np.ceil(H / patch_size))
    n_w = int(np.ceil(W / patch_size))

    # Prepare output buffer
    prediction = np.zeros((H, W), dtype=np.float32)

    tiles = []
    coords = []

    # Create all tiles
    for i in range(n_h):
        for j in range(n_w):
            sh = i * patch_size
            sw = j * patch_size
            eh = min(sh + patch_size, H)
            ew = min(sw + patch_size, W)
            tile = image[sh:eh, sw:ew].astype(np.float32)
            # pad to patch_size with reflect
            pad_h = patch_size - tile.shape[0]
            pad_w = patch_size - tile.shape[1]
            if pad_h > 0 or pad_w > 0:
                tile = np.pad(tile, ((0, pad_h), (0, pad_w)), mode='reflect')
            tiles.append(tile)
            coords.append((sh, eh, sw, ew, pad_h, pad_w))

    if len(tiles) == 0:
        return prediction

    # ensure device string and model device agree; move model only if required
    target_dev = torch.device(device if torch.cuda.is_available() else 'cpu')
    model_dev = next(model.parameters()).device
    if model_dev != target_dev:
        model.to(target_dev)

    total_tiles = len(tiles)
    idx = 0
    while idx < total_tiles:
        start_idx = idx
        batch_tiles = tiles[idx: idx + batch_size]
        idx += len(batch_tiles)  # actual size for this batch
        batch_np = np.stack(batch_tiles, axis=0)  # (B,H,W)

        # normalize per-tile (PerImageZScore behavior)
        batch_t = []
        for t in batch_np:
            m_t, s_t = t.mean(), t.std() + 1e-6
            batch_t.append((t - m_t) / s_t)
        batch_t = np.stack(batch_t, axis=0)
        batch_t = torch.from_numpy(batch_t).float().unsqueeze(1).to(target_dev)  # (B,1,H,W)

        with torch.cuda.amp.autocast(), torch.no_grad():
            logits = model(batch_t)
            probs = torch.sigmoid(logits).cpu().numpy()[:, 0]  # (B,H,W)

        # write back outputs to prediction
        for k, prob in enumerate(probs):
            sh, eh, sw, ew, pad_h, pad_w = coords[start_idx + k]
            h_crop = patch_size - pad_h
            w_crop = patch_size - pad_w
            prob_cropped = prob[:h_crop, :w_crop]
            prediction[sh:eh, sw:ew] = prob_cropped

    return prediction

# -----------------------
# TRAIN / EVAL / TEST FUNCTIONS
# -----------------------

def train_fn(loader, model, optimizer, loss_fn, device, scaler):
    model.train()
    total_loss = 0.0

    loop = tqdm(loader, desc="Training", leave=False)
    for imgs, masks in loop:
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            logits = model(imgs)

            if logits.dim() == 4 and masks.dim() == 3:
                masks = masks.unsqueeze(1)

            loss = loss_fn(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    return total_loss / max(1, len(loader))

@torch.no_grad()
def eval_fn(loader, model, loss_fn, device):
    model.eval()
    total_loss = 0.0

    dice_list = []
    iou_list = []
    pixel_correct = 0
    total_pixels = 0

    for imgs, masks in loader:
        imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)

        logits = model(imgs)

        if logits.dim() == 4 and masks.dim() == 3:
            masks = masks.unsqueeze(1)

        loss = loss_fn(logits, masks)
        total_loss += loss.item()

        preds = torch.sigmoid(logits)
        preds_bin = (preds > 0.45).float()

        pixel_correct += (preds_bin == masks).sum().item()
        total_pixels += masks.numel()

        inter = (preds_bin * masks).sum(dim=(1, 2, 3))
        pred_sum = preds_bin.sum(dim=(1, 2, 3))
        mask_sum = masks.sum(dim=(1, 2, 3))
        union = pred_sum + mask_sum - inter

        dice = (2 * inter + 1e-7) / (pred_sum + mask_sum + 1e-7)
        iou = (inter + 1e-7) / (union + 1e-7)

        dice_list.append(dice.mean().item())
        iou_list.append(iou.mean().item())

    pixel_acc = pixel_correct / total_pixels
    pixel_err = 1 - pixel_acc

    return total_loss / len(loader), pixel_acc, pixel_err, np.mean(dice_list), np.mean(iou_list)

 
metrics = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "pixel_acc": [],
    "pixel_err": [],
    "dice": [],
    "iou": []
}

def save_metrics_csv(path="training_metrics.csv"):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        header = ["epoch", "train_loss", "val_loss", "pixel_acc", "pixel_err", "dice", "iou"]
        writer.writerow(header)
        for i in range(len(metrics["epoch"])):
            writer.writerow([
                metrics["epoch"][i],
                metrics["train_loss"][i],
                metrics["val_loss"][i],
                metrics["pixel_acc"][i],
                metrics["pixel_err"][i],
                metrics["dice"][i],
                metrics["iou"][i]
            ])

def plot_loss_curve(save_path="loss_curve.png"):
    plt.figure(figsize=(8,6))
    plt.plot(metrics["epoch"], metrics["train_loss"], label="Train Loss")
    plt.plot(metrics["epoch"], metrics["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training & Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_metric_curve(save_path="metric_curve.png"):
    plt.figure(figsize=(8,6))
    plt.plot(metrics["epoch"], metrics["dice"], label="Dice")
    plt.plot(metrics["epoch"], metrics["iou"], label="IoU")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Dice & IoU over Epochs")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# Optional: a single combined figure
def plot_combined(save_path="combined_curves.png"):
    fig, ax = plt.subplots(1, 2, figsize=(14,6))
    # Loss
    ax[0].plot(metrics["epoch"], metrics["train_loss"], label="Train Loss")
    ax[0].plot(metrics["epoch"], metrics["val_loss"], label="Val Loss")
    ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].set_title("Loss")
    ax[0].legend(); ax[0].grid(True)
    # Metrics
    ax[1].plot(metrics["epoch"], metrics["dice"], label="Dice")
    ax[1].plot(metrics["epoch"], metrics["iou"], label="IoU")
    ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Score"); ax[1].set_title("Dice & IoU")
    ax[1].legend(); ax[1].grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# -----------------------
# TRAIN LOOP (modified)
# -----------------------
def train_model(train_loader,test_loader,epochs=40, save_checkpoint=True, ckpt_path="BTresunet_model.pth"):
    best_dice = 0.0

    # Clear metrics each run
    for k in metrics.keys():
        metrics[k].clear()

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch [{epoch}/{epochs}]")

        train_loss = train_fn(train_loader, model, optimizer, loss_fn, device, scaler)
        val_loss, pixel_acc, pixel_err, dice, iou = eval_fn(test_loader, model, loss_fn, device)

        scheduler.step(val_loss)

        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Pixel Acc: {pixel_acc*100:.2f}% | Pixel Err: {pixel_err*100:.2f}% | "
              f"Dice: {dice:.4f} | IoU: {iou:.4f}")

        # Log metrics
        metrics["epoch"].append(epoch)
        metrics["train_loss"].append(train_loss)
        metrics["val_loss"].append(val_loss)
        metrics["pixel_acc"].append(pixel_acc)
        metrics["pixel_err"].append(pixel_err)
        metrics["dice"].append(dice)
        metrics["iou"].append(iou)

        # Save best checkpoint
        if save_checkpoint and dice > best_dice:
            best_dice = dice
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'dice': dice
            }, ckpt_path)
            print(f"Saved best model! Dice={dice:.4f}")

        # Save curves every epoch (so you can inspect mid-training)
        save_metrics_csv("training_metrics.csv")
        plot_loss_curve("loss_curve.png")
        plot_metric_curve("metric_curve.png")
        plot_combined("combined_curves.png")

       

    # Final prints and saved artifacts
    print("\nTraining finished.")
    print(f"Saved metrics CSV: training_metrics.csv")
    print(f"Saved loss curve: loss_curve.png")
    print(f"Saved metric curve: metric_curve.png")
    print(f"Saved combined curves: combined_curves.png")

# -----------------------
# QUICK EVAL ON SAVED CHECKPOINT (optional helper)
# -----------------------
def load_and_evaluate(test_loader,ckpt_path="BTresunet_model.pth", loader=None):
    if not os.path.exists(ckpt_path):
        print("Checkpoint not found:", ckpt_path)
        return
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print("Loaded checkpoint from epoch", ckpt.get('epoch', '?'), "dice:", ckpt.get('dice', '?'))
    if loader is None:
        loader = test_loader
    val_loss, pixel_acc, pixel_err, dice, iou = eval_fn(loader, model, loss_fn, device)
    print(f"Eval on loader -> Val Loss: {val_loss:.4f} | Pixel Acc: {pixel_acc*100:.2f}% | Dice: {dice:.4f} | IoU: {iou:.4f}")
    return val_loss, pixel_acc, pixel_err, dice, iou

 


import torch.multiprocessing as mp

def main():

    train_images = sorted(glob("./archive/brisc2025/segmentation_input/train/images/*.jpg"))
    train_masks = sorted(glob("./archive/brisc2025/segmentation_input/train/masks/*.png"))
    test_images = sorted(glob("./archive/brisc2025/segmentation_input/test/images/*.jpg"))
    test_masks = sorted(glob("./archive/brisc2025/segmentation_input/test/masks/*.png"))

    print(f"Number of test images: {len(test_images)}")
    print(f"Number of test masks: {len(test_masks)}")
    print(f"Number of train images: {len(train_images)}")
    print(f"Number of train masks: {len(train_masks)}")


    train_dataset = PatchDataset(train_images, train_masks, transform=train_transform)
    test_dataset = PatchDataset(test_images, test_masks, transform=test_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    train_model(train_loader, test_loader, epochs=40)


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)
    main()



NameError: name 'torch' is not defined